In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/drive/MyDrive/IS/Butterfly Dataset",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224,224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/drive/MyDrive/IS/Butterfly Dataset",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224,224),
    batch_size=32
)

Found 832 files belonging to 10 classes.
Using 666 files for training.
Found 832 files belonging to 10 classes.
Using 166 files for validation.


In [ ]:
base_model = tf.keras.applications.EfficientNetV2B0(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
base_model.trainable = False #freeze layer

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

In [ ]:
tf.keras.layers.Dense(128, activation='relu'),
tf.keras.layers.Dropout(0.5),
tf.keras.layers.Dense(10, activation='softmax')

<Dense name=dense_1, built=False>

In [ ]:
optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001)

In [ ]:
model = tf.keras.Sequential([
    data_augmentation,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

Epoch 1/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 270s 12s/step - accuracy: 0.1299 - loss: 3.2686 - val_accuracy: 0.3072 - val_loss: 2.0925
Epoch 2/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 13s 621ms/step - accuracy: 0.2632 - loss: 2.2847 - val_accuracy: 0.6145 - val_loss: 1.7896
Epoch 3/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 487ms/step - accuracy: 0.4530 - loss: 1.6437 - val_accuracy: 0.8253 - val_loss: 1.5177
Epoch 4/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 12s 548ms/step - accuracy: 0.5990 - loss: 1.2024 - val_accuracy: 0.8976 - val_loss: 1.2834
Epoch 5/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.6749 - loss: 1.0083 - val_accuracy: 0.9337 - val_loss: 1.0785
Epoch 6/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 18s 559ms/step - accuracy: 0.7869 - loss: 0.6966 - val_accuracy: 0.9518 - val_loss: 0.9020
Epoch 7/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 12s 571ms/step - accuracy: 0.8197 - loss: 0.6787 - val_accuracy: 0.9639 - val_loss: 0.7483
Epoch 8/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 12s 563ms/step - accuracy: 0.8311 - loss: 0.5525 - val_accur

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

# โหลดรูป
img = image.load_img("/content/Cabbage White.jpg", target_size=(224,224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

# ทำนาย
prediction = model.predict(img_array)

# หา index ของ class ที่ probability สูงสุด
predicted_index = np.argmax(prediction)

# ชื่อ class
class_names = train_ds.class_names
predicted_class = class_names[predicted_index]

print("Predicted class:", predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Predicted class: Cabbage White


In [ ]:
model.save("butterfly_model.keras")